# Lesson 4: Building a Multi-Document Agent

Extend the research agent across multiple papers with Gemini + LlamaIndex, then scale with **tool retrieval** (`ObjectIndex`).

## Setup

In [2]:
from helper import get_google_api_key, configure_settings

GOOGLE_API_KEY = get_google_api_key()
llm, embed_model = configure_settings()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
import nest_asyncio

nest_asyncio.apply()

## 1. Setup an agent over 3 papers

Place these PDFs in `week_3/` (or download with the URLs below):

- `metagpt.pdf`
- `longlora.pdf`
- `selfrag.pdf`

In [7]:
urls = [
    "https://openreview.net/pdf?id=VtmBAGCN7o",
    "https://openreview.net/pdf?id=6PmJoRfdaK"
]

papers = [
    "metagpt.pdf",
    "longlora.pdf",
]

# Optional download:
# for url, paper in zip(urls, papers):
#     !curl -L "{url}" -o "{paper}"

In [8]:
from pathlib import Path

from utils import get_doc_tools

paper_to_tools_dict = {}
for paper in papers:
    print(f"Getting tools for paper: {paper}")
    vector_tool, summary_tool = get_doc_tools(paper, Path(paper).stem)
    paper_to_tools_dict[paper] = [vector_tool, summary_tool]

Getting tools for paper: metagpt.pdf
Getting tools for paper: longlora.pdf


In [9]:
initial_tools = [t for paper in papers for t in paper_to_tools_dict[paper]]

In [10]:
from llama_index.llms.google_genai import GoogleGenAI
from helper import DEFAULT_LLM_MODEL

llm = GoogleGenAI(model=DEFAULT_LLM_MODEL, api_key=GOOGLE_API_KEY, temperature=0)

2026-07-14 09:47:32,447 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it "HTTP/1.1 200 OK"


In [11]:
len(initial_tools)

4

In [12]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.workflow import Context

agent = FunctionAgent(
    tools=initial_tools,
    llm=llm,
    system_prompt=(
        "You are an agent designed to answer queries over a set of given papers. "
        "Please always use the tools provided to answer a question. "
        "Do not rely on prior knowledge."
    ),
    verbose=True,
)
ctx = Context(agent)

In [13]:
response = await agent.run(
    "Tell me about the evaluation dataset used in LongLoRA, "
    "and then tell me about the evaluation results",
    ctx=ctx,
)
print(str(response))

2026-07-14 09:47:35,991 - INFO - [tick] add: AgentWorkflowStartEvent(user_msg='Tell me about the evaluation dataset used in LongL...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
2026-07-14 09:47:35,991 - INFO - [init_run:0] started from AgentWorkflowStartEvent
2026-07-14 09:47:36,246 - INFO - [init_run:0] complete with AgentInput
2026-07-14 09:47:36,246 - INFO - [tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Tell me about the evaluation dataset used in LongLoRA, and then tell ...
2026-07-14 09:47:36,247 - INFO - [setup_agent:0] started from AgentInput
2026-07-14 09:47:36,247 - INFO - [setup_agent:0] complete with AgentSetup
2026-07-14 09:47:36,248 - INFO - [tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='You are an agent designed to answer queries over a set of gi

The evaluation of LongLoRA utilized the **proof-pile test set** and the **PG19 test split**.

The evaluation results are summarized as follows:

### Long-sequence Language Modeling (Perplexity)
Perplexity generally decreased as the context size increased across different Llama2 models:
*   **Llama2 7B:** Increasing the context window from 8,192 to 32,768 improved perplexity from 2.72 to 2.50. With a training context length of 100,000, perplexity ranged from 3.36 (at 2,048 evaluation context length) to 2.52 (at 100,000 evaluation context length). On the PG19 test split, results ranged from 8.38 to 7.04.
*   **Llama2 13B:** Increasing the context window from 8,192 to 32,768 reduced perplexity by 0.28. With a training context length of 65,536, perplexity ranged from 3.20 (at 2,048 evaluation context length) to 2.38 (at 65,536 evaluation context length). On the PG19 test split, results ranged from 7.63 to 6.29.
*   **Llama2 70B:** With a training context length of 32,768, perplexity ranged

In [14]:
response = await agent.run(
    "Give me a summary of both Self-RAG and LongLoRA",
    ctx=ctx,
)
print(str(response))

2026-07-14 09:48:41,421 - INFO - [tick] add: AgentWorkflowStartEvent(user_msg='Give me a summary of both Self-RAG and LongLoRA', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
2026-07-14 09:48:41,422 - INFO - [init_run:0] started from AgentWorkflowStartEvent
2026-07-14 09:48:41,564 - INFO - [init_run:0] complete with AgentInput
2026-07-14 09:48:41,564 - INFO - [tick] add: AgentInput(input=[6 items], current_agent_name='Agent')
2026-07-14 09:48:41,564 - INFO - [setup_agent:0] started from AgentInput
2026-07-14 09:48:41,575 - INFO - [setup_agent:0] complete with AgentSetup
2026-07-14 09:48:41,576 - INFO - [tick] add: AgentSetup(input=[7 items], current_agent_name='Agent')
2026-07-14 09:48:41,576 - INFO - [run_agent_step:0] started from AgentSetup
2026-07-14 09:49:08,521 - INFO - [run_agent_step:0] complete with AgentOutput
2026-07-14 09:49:08,521 - INFO - [tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_k

Here is a summary of **LongLoRA**. Please note that I do not have access to the **Self-RAG** paper, but I can provide information on **MetaGPT** if that would be helpful.

### **LongLoRA Summary**
LongLoRA is an efficient fine-tuning approach designed to extend the context windows of pre-trained large language models (LLMs) while minimizing computational costs. It addresses the quadratic increase in self-attention costs associated with long contexts through two primary innovations:

*   **Shifted Sparse Attention (S2-Attn):** During fine-tuning, LongLoRA replaces dense global attention with sparse local attention. It splits the context into groups and conducts attention within each group. To maintain information flow between neighboring groups, tokens in half of the attention heads are shifted by half the group size. This allows the model to retain its original standard self-attention architecture during inference, ensuring compatibility with optimizations like Flash-Attention2.
*   **

## 2. Setup an agent over 11 papers

### Download 11 ICLR papers

When scaling to many tools, pass a **tool retriever** instead of all tools at once.

In [16]:
urls = [
    "https://openreview.net/pdf?id=VtmBAGCN7o",
    "https://openreview.net/pdf?id=6PmJoRfdaK",
    "https://openreview.net/pdf?id=LzPWWPAdY4",
    "https://openreview.net/pdf?id=VTF8yNQM66",
    "https://openreview.net/pdf?id=hSyW5go0v8",
    "https://openreview.net/pdf?id=9WD9KwssyT",
    "https://openreview.net/pdf?id=yV6fD7LYkF",
    "https://openreview.net/pdf?id=hnrB5YHoYu",
    "https://openreview.net/pdf?id=WbWtOYIzIK",
    "https://openreview.net/pdf?id=c5pwL0Soay",
    "https://openreview.net/pdf?id=TpD2aG1h0D",
]

papers = [
    "metagpt.pdf",
    "longlora.pdf"
]

# Optional download:
# for url, paper in zip(urls, papers):
#     !curl -L "{url}" -o "{paper}"

In [17]:
from pathlib import Path

from utils import get_doc_tools

paper_to_tools_dict = {}
for paper in papers:
    print(f"Getting tools for paper: {paper}")
    vector_tool, summary_tool = get_doc_tools(paper, Path(paper).stem)
    paper_to_tools_dict[paper] = [vector_tool, summary_tool]

Getting tools for paper: metagpt.pdf
Getting tools for paper: longlora.pdf


### Extend the Agent with Tool Retrieval

In [18]:
all_tools = [t for paper in papers for t in paper_to_tools_dict[paper]]

In [20]:
paper_to_tools_dict

{'metagpt.pdf': [<llama_index.core.tools.function_tool.FunctionTool at 0x12ca7da30>,
 'longlora.pdf': [<llama_index.core.tools.function_tool.FunctionTool at 0x31418f6e0>,
  <llama_index.core.tools.query_engine.QueryEngineTool at 0x3141e27e0>]}

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.core.objects import ObjectIndex

obj_index = ObjectIndex.from_objects(
    all_tools,
    index_cls=VectorStoreIndex,
)

In [ ]:
obj_retriever = obj_index.as_retriever(similarity_top_k=3)

In [ ]:
tools = obj_retriever.retrieve(
    "Tell me about the eval dataset used in MetaGPT and SWE-Bench"
)
for t in tools:
    print(t.metadata)

In [ ]:
agent = FunctionAgent(
    tool_retriever=obj_retriever,
    llm=llm,
    system_prompt="""\
You are an agent designed to answer queries over a set of given papers.
Please always use the tools provided to answer a question. Do not rely on prior knowledge.
""",
    verbose=True,
)
ctx = Context(agent)

In [ ]:
response = await agent.run(
    "Tell me about the evaluation dataset used "
    "in MetaGPT and compare it against SWE-Bench",
    ctx=ctx,
)
print(str(response))

In [ ]:
response = await agent.run(
    "Compare and contrast the LoRA papers (LongLoRA, LoftQ). "
    "Analyze the approach in each paper first.",
    ctx=ctx,
)
print(str(response))